# A 10,000-neuron V1 laminar column in jaxfne: spectrolaminar proxy readouts

Companion notebook for `report.tex`. Builds a single V1 laminar column of 10,000 reduced
Izhikevich emitters, simulates 1000 ms at dt=0.1 ms, and reads out the laminar
field with the package **spectrolaminar suite**.

Deep layers (L5/L6) are sparser and more excitatory and receive a tunable DC drive; the
dense recurrent connectivity carries deep activity to superficial layers (where faster PV
inhibition supports relatively higher gamma).

**Scope:** all field outputs are proxy readouts — `field_solver_status=linear_solver`,
`field_claim_level=proxy_readout`, `physical_amplitude_calibrated=False`. Not physical
EEG/MEG/LFP/CSD measurements.

In [ ]:
import numpy as np
import jax.numpy as jnp
import jaxfne as jtfne

N, DUR, DT = 10000, 1000.0, 0.1
LAYERS = ["L1", "L2/3", "L4", "L5", "L6"]
# deep layers thin (sparser) + positioned deep
LAYER_RANGES = {"L1": (0.0, 0.12), "L2/3": (0.12, 0.46), "L4": (0.46, 0.66),
                "L5": (0.66, 0.80), "L6": (0.80, 1.0)}
# deep layers more E; superficial more PV (fast inhibition -> gamma)
LAYER_CT = {"L1": {"E": 0.5, "PV": 0.2, "SST": 0.2, "VIP": 0.1},
            "L2/3": {"E": 0.72, "PV": 0.15, "SST": 0.09, "VIP": 0.04},
            "L4": {"E": 0.78, "PV": 0.13, "SST": 0.07, "VIP": 0.02},
            "L5": {"E": 0.90, "PV": 0.06, "SST": 0.03, "VIP": 0.01},
            "L6": {"E": 0.92, "PV": 0.05, "SST": 0.02, "VIP": 0.01}}
DEEP = {"L5", "L6"}

## Build the V1 column (Config -> construct)

In [ ]:
def build(deep_dc=0.0):
    cfg = jtfne.laminar_cortex_config(areas=["V1"], layers=LAYERS, n=N,
                                      duration_ms=DUR, dt_ms=DT, seed=0)
    cfg = cfg.layer_fractions(layer_fractions=dict(LAYER_RANGES))
    cfg = cfg.area_layer_cell_types("V1", LAYER_CT)
    model = jtfne.construct(cfg)
    nt = model.neuron_table()
    deep_mask = np.array([r["layer"] in DEEP for r in nt], dtype="float32")
    emit = model.params["emitter"]
    drive = np.asarray(emit.drive) + deep_mask * deep_dc  # DC to all deep cells
    model = model.with_emitter_parameters(drive_per_neuron=jnp.asarray(drive, dtype=emit.drive.dtype))
    return model

model = build(deep_dc=10.0)
print("built", N, "neurons")

## Cell densities per layer (fig 10)

In [ ]:
from collections import Counter
nt = model.neuron_table()
for L in LAYERS:
    rows = [r for r in nt if r["layer"] == L]
    ef = np.mean([r["cell_type"] == "E" for r in rows]) if rows else 0
    print(f"{L:5s} n={len(rows):5d}  E-frac={ef:.2f}")

## Simulate (Emitter -> Source) and the spectrolaminar suite (figs 1,2,7,8)

In [ ]:
sig = jtfne.simulate(model, duration_ms=DUR, dt_ms=DT, seed=0)
rate = float(jnp.sum(sig.get("spk")) / N / (DUR / 1000.0))
print("mean rate (Hz):", round(rate, 2),
      "| vm finite:", bool(jnp.all(jnp.isfinite(sig.get("vm")))))
jtfne.vis.spectrolaminar_suite(sig)   # fig 2 — preferred readout

In [ ]:
jtfne.vis.lfp(sig)          # fig 1 — LFP proxy
jtfne.vis.csd(sig)          # fig 7 — CSD proxy
jtfne.vis.rate(sig)         # fig 8 — firing rate
jtfne.vis.raster(sig)       # raster

## EEG and MEG proxies (figs 5,6)

In [ ]:
jtfne.vis.eeg(sig)   # fig 5 — eeg_proxy
jtfne.vis.meg(sig)   # fig 6 — meg_proxy

## Rasters & spectrolaminar suites across deep-drive levels (figs 3,4)
Rebuild is only needed if layer structure changes; here we reapply per-neuron DC and resimulate.

In [ ]:
for dc in [0.0, 5.0, 10.0]:
    m = build(deep_dc=dc)
    s = jtfne.simulate(m, duration_ms=DUR, dt_ms=DT, seed=0)
    print(f"deep_DC={dc}")
    jtfne.vis.raster(s)
    jtfne.vis.spectrolaminar_suite(s)

## 3D circuit of the 10,000-neuron column (fig 9, interactive Plotly)

In [ ]:
jtfne.vis.visualize_network_3d(model.neuron_table(),
    title="V1 column — 10,000 neurons", show_layers=True,
    coordinate_unit="mm", display_unit="um",
    output_html="figs/09_circuit3d_10k.html")

## Manifest (JSON-safe, schema-tagged)

In [ ]:
manifest = jtfne.manifest(model.cfg, sig)
print("claim_level:", manifest.get("claim_level"))
print("field_solver_status:", manifest.get("field_solver_status"))
print("field_claim_level:", manifest.get("field_claim_level"))
print("physical_amplitude_calibrated:", manifest.get("physical_amplitude_calibrated"))